
# Packages import

In [1]:
import os
import yaml
import requests
import pandas as pd
from datetime import datetime
import uuid
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup

# Apollo scraper

In [4]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['username']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [6]:

group_name = input("Enter group ID: ")
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

200


In [7]:

print(response.text.encode().decode())

<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN" "http://www.w3.org/TR/REC-html40/loose.dtd">
<html>
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
<title>Plan zajęć UEK ZICSS1-1211</title>
<link rel="stylesheet" type="text/css" href="planzajec.css">
</head>
<body>
<script src="js/accessibility_uek.js"></script><script src="js/apollo2calendar.js"></script><div class="naglowek">
<div class="logo"><img src="UEK-logo.gif" alt=""></div>
<div class="planzajec"><a href="index.php">Plan Zajęć</a></div>
</div>
<form action="index.php" method="get">
<input type="hidden" name="typ" value="G"><input type="hidden" name="id" value="252681">okres: <select name="okres"><option value="1">2026-05-25 do 2026-06-08</option>
<option selected value="2">2026-02-23 do 2026-09-30</option>
<option value="3">2025-10-01 do 2026-02-22</option>
<option value="4">2024-10-01 do 2025-09-30</option></select><button type="submit">Ok</button>
</form>
<div class="grupa">ZICSS1-121

In [8]:

page_dom = BeautifulSoup(response.text, "html.parser")

In [9]:

group = page_dom.select_one("div.grupa").get_text(strip=True)
print(group)

ZICSS1-1211


In [10]:

classes_tag = page_dom.select_one("table")
with open("temp.html", "w", encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [11]:
classes = classes.loc[
    classes['Typ'].isin(["ćwiczenia", "wykład", "egzamin"])
]

In [12]:
classes[["Day", "Start time", "hyphen", "End time", "Duration"]] = classes["Dzień, godzina"].str.split(" ", expand=True)

In [13]:

classes["Duration"] = classes["Duration"].map(lambda x: x.split('(')[1].split("g")[0])

In [14]:

classes = classes.drop(['Dzień, godzina', 'hyphen'], axis=1)

In [15]:

classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.).*",
    r"\1",
    regex=True
)

In [16]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [17]:

classes.to_csv(f"schedules/{group}.csv")

In [18]:

classes

,Termin,Przedmiot,Typ,Nauczyciel,Sala,Day,Start time,End time,Duration
1,2026-02-23,Computer Programming 2,ćwiczenia,dr Katarzyna Wójcik,Paw.A 013 lab.,Pn,13:15,15:45,3
4,2026-02-24,Probability and Statistics,wykład,prof. dr hab. Andrzej Sokołowski,Paw.F 008,Wt,09:45,11:15,2
6,2026-02-24,Probability and Statistics,ćwiczenia,prof. dr hab. Andrzej Sokołowski,Paw.A 07 lab.,Wt,13:15,14:45,2
8,2026-02-24,Discrete mathematics,wykład,dr Grzegorz Kosiorowski,Rakowicka 16 sala 11,Wt,15:00,16:30,2
9,2026-02-24,Business Law,wykład,dr Jacek Lachner,Rakowicka 16 sala 11,Wt,18:30,20:00,2
...,...,...,...,...,...,...,...,...,...
304,2026-06-11,Information Systems,ćwiczenia,mgr inż. Justyna Olczak,Paw.A 014 lab.,Cz,16:45,17:30,1
305,2026-06-11,Operating Systems and Computer Networks,ćwiczenia,prof. UEK dr hab. Joanna Wyrobek,Paw.A 07 lab.,Cz,18:30,20:00,2
307,2026-06-17,Discrete mathematics,egzamin,dr Grzegorz Kosiorowski,Paw.C Nowa Aula,Śr,10:30,13:00,3
308,2026-06-26,Operating Systems and Computer Networks,egzamin,prof. UEK dr hab. Joanna Wyrobek,Paw.A 07 lab.,Pt,10:30,12:15,2


In [ ]:
def escape_ics(text):
    if text is None:
        return ""
    return str(text).replace("\\", "\\\\").replace(",", "\\,").replace(";", "\\;").replace("\n", "\\n")

ics = [
    "BEGIN:VCALENDAR",
    "VERSION:2.0",
    "PRODID:-//WebScraper//Plan zajec//PL",
    "CALSCALE:GREGORIAN"
]

for _, row in classes.iterrows():
    start = datetime.strptime(
        f"{row['Termin']} {row['Start time']}",
        "%Y-%m-%d %H:%M"
    )
    end = datetime.strptime(
        f"{row['Termin']} {row['End time']}",
        "%Y-%m-%d %H:%M"
    )

    title = f"{row['Przedmiot']} - {row['Typ']}"
    location = row["Sala"]
    description = f"Nauczyciel: {row['Nauczyciel']}"

    ics += [
        "BEGIN:VEVENT",
        f"UID:{uuid.uuid4()}",
        f"DTSTAMP:{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}",
        f"DTSTART;TZID=Europe/Warsaw:{start.strftime('%Y%m%dT%H%M%S')}",
        f"DTEND;TZID=Europe/Warsaw:{end.strftime('%Y%m%dT%H%M%S')}",
        f"SUMMARY:{escape_ics(title)}",
        f"LOCATION:{escape_ics(location)}",
        f"DESCRIPTION:{escape_ics(description)}",
        "END:VEVENT"
    ]

ics.append("END:VCALENDAR")

with open("plan_zajec.ics", "w", encoding="utf-8") as f:
    f.write("\n".join(ics))

print("Zapisano plik plan_zajec.ics")